#### **Data Ingestion Pipeline to Query Retrieval Pipeline**

In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\bijsh\AppData\Local\Temp\ipykernel_46468\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [16]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Bandyopadhyay et al. - 2023 - Alzheimer’s Disease Detection Using Ensemble Learn.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/ELVRBN+TimesNewRomanPS-BoldItalicMT', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 97, '/FontDescriptor': IndirectObject(38, 0, 2478077738880), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [500, 0, 444, 0, 444, 0, 500, 0, 278, 0, 0, 278, 0, 0, 0, 0, 0, 389, 0, 278, 556]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


  ✓ Loaded 11 pages

Processing: Moysiadis et al. - 2023 - Monitoring Mushroom Growth with Machine Learning.pdf
  ✓ Loaded 17 pages

Processing: Patel - 2023 - An Overview and Application of Deep Convolutional .pdf
  ✓ Loaded 7 pages

Total documents loaded: 35


In [17]:
type(all_pdf_documents), len(all_pdf_documents)

(list, 35)

In [18]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [19]:
chunks=split_documents(all_pdf_documents)
chunks

Split 35 documents into 137 chunks

Example chunk:
Content: See discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/367033952
Alzheimer’s Disease Detection Using Ensemble Learning and Artiﬁcial Neural
Netwo...
Metadata: {'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-14T20:22:41+00:00', 'author': '', 'keywords': '', 'moddate': '2023-01-14T20:22:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'subject': '', 'title': '', 'trapped': '/False', 'rgid': 'PB:367033952_AS:11431281113168281@1673727819724', 'source': '..\\data\\pdf\\Bandyopadhyay et al. - 2023 - Alzheimer’s Disease Detection Using Ensemble Learn.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Bandyopadhyay et al. - 2023 - Alzheimer’s Disease Detection Using Ensemble Learn.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.20', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-14T20:22:41+00:00', 'author': '', 'keywords': '', 'moddate': '2023-01-14T20:22:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathsea version 6.3.1', 'subject': '', 'title': '', 'trapped': '/False', 'rgid': 'PB:367033952_AS:11431281113168281@1673727819724', 'source': '..\\data\\pdf\\Bandyopadhyay et al. - 2023 - Alzheimer’s Disease Detection Using Ensemble Learn.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Bandyopadhyay et al. - 2023 - Alzheimer’s Disease Detection Using Ensemble Learn.pdf', 'file_type': 'pdf'}, page_content='See discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/367033952\nAlzheimer’s Disease Detection Using Ensemble Learning and Artiﬁcial Neural\nNetworks\nChapter · January 2023\nDOI: 10.1007/978-3-031-23599-3_2\nCITATIONS\n2\nREADS\

#### **Embedding and VectorStore DB**

In [1]:
%pip install chromadb scikit-learn

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl.metadata (2.6 kB)
Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl (23.5 MB)
Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl (19 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install sentence-transformers

  Using cached sentence_transformers-6.1.0-py3-none-any.whl.metadata (20 kB)
Using cached sentence_transformers-6.1.0-py3-none-any.whl (740 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Bijay\University Documents (DSU)\Research Files\RAG\rag-bijay\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class EmbeddingManager:
    """Handle embedding generation and storage using ChromaDB and Sentence Transformers."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager with a specific model."""
        self.model_name = model_name
        self.model = None
        self._load_model() ##protected method to load the model

    
    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
    
## intialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2777.42it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\bijsh\AppData\Local\Temp\ipykernel_46468\2363640183.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


#### **Vector Store using ChromaDB**

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        ## Loop through documents and embeddings to prepare for insertion --> Tuple of (document, embedding) is enumerated to generate unique IDs and metadata (immutable)
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [20]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store embeddings in the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 137 texts...


Batches: 100%|██████████| 5/5 [00:04<00:00,  1.13it/s]


Generated embeddings with shape: (137, 384)
Adding 137 documents to vector store...
Successfully added 137 documents to vector store
Total documents in collection: 137


#### **Retriever Pipeline from Vector Store**

In [31]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    print(f"Document ID: {doc_id}, Similarity Score: {similarity_score}, Distance: {distance}")
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [22]:
rag_retriever

In [30]:
print(vectorstore.collection.count())

137


In [33]:
rag_retriever.retrieve("Explain Alzheimer's disease?")

Retrieving documents for query: 'Explain Alzheimer's disease?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.07it/s]

Generated embeddings with shape: (1, 384)
Document ID: doc_a13506ef_3, Similarity Score: 0.3512081503868103, Distance: 0.6487918496131897
Document ID: doc_1f36832b_13, Similarity Score: 0.03361082077026367, Distance: 0.9663891792297363
Document ID: doc_bdc98f21_12, Similarity Score: -0.029238343238830566, Distance: 1.0292383432388306
Document ID: doc_daafc309_2, Similarity Score: -0.07159161567687988, Distance: 1.0715916156768799
Document ID: doc_d0b7f4c5_28, Similarity Score: -0.0860898494720459, Distance: 1.086089849472046
Retrieved 2 documents (after filtering)


[{'id': 'doc_a13506ef_3',
  'content': 'Keywords: Alzheimer’s disease detection· Machine learning· ANN · Ensemble\nlearning\n1 Introduction\n1.1 Context\nAlzheimer’s disease (AD) is a type of dementia that aﬀects a person’s mem-\nory, thought, and behaviour. It is a disease that begins with mildly aﬀecting\nparts of the brain, which makes an individual have diﬃculty in remembering\nany newly learned information, constant changes in mood, and confusion with\nevents, times, and places. AD usually starts in an individual after the age of60\nyears and its risk increases as the person ages. So the early detection of the dis-\nease can help the doctors start the treatment which can signiﬁcantly help allevi-\nate some symptoms, reduce their intensity, and thus contribute to a better qual-\nity of life for patients and their families. In recent years, various MRI biomarkers',
  'metadata': {'page': 1,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019) kpathse

In [34]:
rag_retriever.retrieve("Ensemeble Learning for Alzheimer's disease classification.")

Retrieving documents for query: 'Ensemeble Learning for Alzheimer's disease classification.'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.26it/s]

Generated embeddings with shape: (1, 384)
Document ID: doc_daafc309_2, Similarity Score: 0.22159606218338013, Distance: 0.7784039378166199
Document ID: doc_ff2307f8_1, Similarity Score: 0.1933489441871643, Distance: 0.8066510558128357
Document ID: doc_5959bf4d_29, Similarity Score: 0.11251676082611084, Distance: 0.8874832391738892
Document ID: doc_db6604dd_24, Similarity Score: 0.09695661067962646, Distance: 0.9030433893203735
Document ID: doc_a300d9e6_25, Similarity Score: 0.078055739402771, Distance: 0.921944260597229
Retrieved 5 documents (after filtering)


[{'id': 'doc_daafc309_2',
  'content': '(OASIS) dataset for Alzheimer’s disease (AD) detection. The proposed\nwork performs an in-depth feature examination and a training-test split\nin a 70 : 30 ratio on the dataset and applies 8 diﬀerent ML algorithms.\nThe AD detection outcome is obtained using two procedures, ﬁrst by an\nensemble approach applied to diﬀerent machine learning algorithms, and\nsecondly by using an artiﬁcial neural network (ANN). The use of ANN\nachieves an overall test accuracy of 0.9196 whereas two ensemble tech-\nniques, namely gradient boosting and voting classiﬁer achieve an over-\nall test accuracy of 0.857 and 0.8304. The precision and sensitivity scores\ndemonstrate the superior detection performance of the ANN over the en-\nsemble method on ML algorithms.\nKeywords: Alzheimer’s disease detection· Machine learning· ANN · Ensemble\nlearning\n1 Introduction\n1.1 Context\nAlzheimer’s disease (AD) is a type of dementia that aﬀects a person’s mem-',
  'metadata': {

#### **RAG Pipeline - VectorDB to LLM Output Generation**

In [ ]:

from dotenv import load_dotenv; load_dotenv()

In [48]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)


In [51]:
def rag_with_llm(query: str, retriever: RAGRetriever, llm: ChatGroq, top_k: int=3) -> str:
    """
    Perform RAG with LLM to answer a query.
    
    Args:
        query: The search query
        retriever: RAGRetriever instance for document retrieval
        llm: ChatGroq instance for generating answers
        
    Returns:
        Generated answer from the LLM
    """
    # Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    
    if not context:
        return "No relevant documents found to answer the question."
    
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    print(f"Prompt sent to LLM:\n{prompt}\n")
    # Generate answer using LLM
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    return response.content


In [52]:
answer=rag_with_llm("What is Mushroom Growth Stage Detection?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Mushroom Growth Stage Detection?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 96.03it/s]

Generated embeddings with shape: (1, 384)
Document ID: doc_57e1da91_80, Similarity Score: 0.5140872597694397, Distance: 0.4859127402305603
Document ID: doc_16e6a6cf_31, Similarity Score: 0.4821215271949768, Distance: 0.5178784728050232
Document ID: doc_d7879b8a_49, Similarity Score: 0.4468313455581665, Distance: 0.5531686544418335
Retrieved 3 documents (after filtering)
Prompt sent to LLM:
Use the following context to answer the question concisely.
        Context:
        such as a greenhouse with Pleurotus mushrooms, it was possible to identify mushrooms
that were ready to harvest. The evaluation results on mushroom detection and classiﬁcation
in three different growing stages gave an F1-score of up to 76.5%, and speciﬁcally for the
ﬁnal growing stage, an accuracy of up to 70%.
Secondly, we proposed a method for mushroom growth monitoring. For that purpose,
we used two trained models with Detectron2. The ﬁrst one detected the substrate grow bag,
and the second detected the mushrooms 

Mushroom growth‑stage detection is the computer‑vision task of locating mushrooms in images and automatically assigning each one to a predefined developmental class (e.g., Stage 1, Stage 2, Stage 3). By detecting the mushroom and classifying its current growth stage, the system can monitor development, predict when the fruiting bodies will reach maximum size, and help schedule optimal harvest times.


#### **Advanced RAG PIPELINE Features**

In [53]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [55]:
# Example usage:
result = rag_advanced("What is Ensemble Learning?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:500])

Retrieving documents for query: 'What is Ensemble Learning?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.40it/s]


Generated embeddings with shape: (1, 384)
Document ID: doc_987ca6eb_16, Similarity Score: 0.1373412013053894, Distance: 0.8626587986946106
Document ID: doc_cfcac0d6_17, Similarity Score: -0.05816233158111572, Distance: 1.0581623315811157
Document ID: doc_ff2307f8_1, Similarity Score: -0.06476819515228271, Distance: 1.0647681951522827
Retrieved 1 documents (after filtering)
Answer: Ensemble learning is a machine‑learning approach that combines the predictions of multiple models (e.g., decision‑tree stumps in AdaBoost, larger trees in Gradient Boosting, or diverse classifiers in a Voting Classifier) to produce a single, more accurate and robust result, thereby reducing over‑fitting and improving overall performance.
Sources: [{'source': 'Bandyopadhyay et al. - 2023 - Alzheimer’s Disease Detection Using Ensemble Learn.pdf', 'page': 5, 'score': 0.1373412013053894, 'preview': 'and sensitivity. Another ML framework, Adaboost combines multiple stumps\nwho are technically weak learners thus in

In [58]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    """
    Advanced RAG Pipeline with features like streaming, citations, history, and summarization.
    """
    def __init__(self, retriever: RAGRetriever, llm: ChatGroq):
        self.retriever = retriever
        self.llm = llm
        self.history: List[Dict[str, Any]] = []

    def query(self, query: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        """
        Process a query through the RAG pipeline.
        
        Args:
            query: The search query
            top_k: Number of top results to retrieve
            min_score: Minimum similarity score threshold
            stream: Whether to stream the response
            summarize: Whether to summarize the context
            
        Returns:
            Dictionary containing answer, sources, confidence score, and context.
        """
        print(f"Processing query: {query}")
        results = self.retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
        
        if not results:
            return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:300] + '...'
            } for doc in results]
            confidence = max([doc['similarity_score'] for doc in results])
            
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""

            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=query)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
        
        # Store query history
        self.history.append({
            'question': query,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })
        return {
            'question': query,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }


In [ ]:
# Example usage:
#Rag_Retriever(vectorstore, embedding_manager)
#RAGPipeline(Rag_Retriever, llm)

adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Ensemble Learning?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Processing query: What is Ensemble Learning?
Retrieving documents for query: 'What is Ensemble Learning?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.39it/s]

Generated embeddings with shape: (1, 384)
Document ID: doc_987ca6eb_16, Similarity Score: 0.1373412013053894, Distance: 0.8626587986946106
Document ID: doc_cfcac0d6_17, Similarity Score: -0.05816233158111572, Distance: 1.0581623315811157
Document ID: doc_ff2307f8_1, Similarity Score: -0.06476819515228271, Distance: 1.0647681951522827
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
and sensitivity. Another ML framework, Adaboost combines multiple stumps
who are technically weak learners thus increasing their learning strength. The
number of stumps was set to60 signifying decision trees have one node and two
leaves. Adaboost assi

gns equal sample weights to all records which are then fed
into a model and based on the performance of each stump, weights of incorrectly
classiﬁed records are increased and correctly classiﬁed are decreased. Gradient
boosting is an ensemble boosting technique that basically optimizes the classiﬁ-
cation loss function. Decision trees used in gradient boosting are comparatively
larger in size than Adaboost. Voting Classiﬁer uses the technique of ensemble
learning using which the results of various ML models operating simultane-
ously are merged together. This helps in reducing over-ﬁtting and improving
accuracy. Therefore, it combined the outputs of machine learning algorithms

Question: What is Ensemble Learning?

Answer:

Final Answer: Ensemble learning is a machine‑learning approach that combines the predictions of multiple models (e.g., decision trees, stumps, or different algorithms) to produce a single, more accurate and robust result, often reducing over‑fitting and improving ov

In [61]:
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What Alzheimer’s Disease Detection does?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Processing query: What Alzheimer’s Disease Detection does?
Retrieving documents for query: 'What Alzheimer’s Disease Detection does?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 92.16it/s]

Generated embeddings with shape: (1, 384)
Document ID: doc_a13506ef_3, Similarity Score: 0.525185614824295, Distance: 0.47481438517570496
Document ID: doc_daafc309_2, Similarity Score: 0.2692398428916931, Distance: 0.7307601571083069
Document ID: doc_a300d9e6_25, Similarity Score: 0.1520169973373413, Distance: 0.8479830026626587
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Keywords: Alzheimer’s disease detection· Machine learning· ANN · Ensemble
learning
1 Introduction
1.1 Context
Alzheimer’s disease (AD) is a type of dementia that aﬀects a person’s mem-
ory, thought, and behaviour. It is a disease that begins with mild

ly aﬀecting
parts of the brain, which makes an individual have diﬃculty in remembering
any newly learned information, constant changes in mood, and confusion with
events, times, and places. AD usually starts in an individual after the age of60
years and its risk increases as the person ages. So the early detection of the dis-
ease can help the doctors start the treatment which can signiﬁcantly help allevi-
ate some symptoms, reduce their intensity, and thus contribute to a better qual-
ity of life for patients and their families. In recent years, various MRI biomarkers

(OASIS) dataset for Alzheimer’s disease (AD) detection. The proposed
work performs an in-depth feature examination and a training-test split
in a 70 : 30 ratio on the dataset and applies 8 diﬀerent ML algorithms.
The AD detection outcome is obtained using two procedures, ﬁrst by an
ensemble approach applied to diﬀerent machine learning algorithms, and
secondly by using an artiﬁcial neural network (ANN). The use of ANN
a